# Volatility Factor: GARCH VaR run

Settings come from `configs/var_config_volfactor.yml`, found automatically in this folder or
any parent folder. To use a different config or price file (same format, different data):

```python
vf.set_config("path/to/config.yml")
vf.set_data("path/to/prices.csv")   # or a .parquet file, or a DataFrame
```

In [ ]:
import pandas as pd
import volatility_factor as vf

cfg = vf.get_config()
print(cfg.path)
print(cfg.garch)
print("Factors:", list(cfg.factors))
print("Output:", cfg.output_file)

**Create GARCH VaR results**

Runs every factor, liquidation period, sample window and starting-value quantile in the
config, then saves `<output_file>.xlsx`, `<output_file>.parquet` (one row per VaR quantile)
and `QA1_<output_file>.xlsx` (windows skipped because prices don't cover the sample).

In [ ]:
garch_run = vf.run_garch_var(cfg).save()
garch_run.results.head()

In [ ]:
garch_run.qa

**Create Circular Bootstrap VaR results** (most recent sample window only)

In [ ]:
boot_run = vf.run_circular_bootstrap(cfg, nboot=10_000, var_p=0.99).save()
boot_run.results

**Ad-hoc simulation**

In [ ]:
ndays = 5
res = (
    vf.VarData("M0A0")
    .garch(sample_start="2016-01-01", sample_end="2024-12-31")
    .simulate(horizon1=63, horizon2=252 + ndays - 1, ndays=ndays, nsims=10_000,
              shock_type="custom_normal", start_vals=("2024-01-01", 0.5))
)
res.to_dict()

In [ ]:
res.plot_distribution()

**Charts**: rolling annualised volatility

In [ ]:
vf.VarData("CMOS").plot_rolling_volatility(
    window=30,
    averages={"2024 - Avg": ("2024-01-01", None), "Pre2022 Avg": (None, "2021-12-31")},
    start_date="2006-01-01", end_date="2024-12-31",
    title="CMO Annualized Volatility - Rolling",
)

In [ ]:
vf.VarData("M0A0").plot_rolling_volatility(
    window=30,
    averages={"2024 - Avg": ("2024-01-01", None), "Pre2022 Avg": (None, "2021-12-31")},
    start_date="2006-01-01", end_date="2024-12-31",
    title="PT MBS Annualized Volatility - Rolling",
)